# Preprocessing Pipeline — Sans Data Leakage

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from src.preprocessing import (
    preprocess_data, build_preprocessor, load_and_split,
    get_feature_names, compute_scale_pos_weight,
    NUMERICAL_FEATURES, CATEGORICAL_FEATURES
)

## Chargement et split stratifié

In [ ]:
X_train, X_test, y_train, y_test, df = load_and_split('../data/raw/customer_churn_business_dataset.csv', task='classification')
print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"Churn train: {y_train.mean():.3f} | Churn test: {y_test.mean():.3f}")
print(f"scale_pos_weight: {compute_scale_pos_weight(y_train):.2f}")
print(f"\nFeatures numériques ({len(NUMERICAL_FEATURES)}):", NUMERICAL_FEATURES)
print(f"\nFeatures catégorielles ({len(CATEGORICAL_FEATURES)}):", CATEGORICAL_FEATURES)

## Application du pipeline — Classification

In [ ]:
X_train_proc, X_test_proc, y_tr, y_te, preprocessor = preprocess_data(
    '../data/raw/customer_churn_business_dataset.csv', task='classification', save=True
)
feature_names = get_feature_names(preprocessor)
print(f"Nombre de features après encodage: {len(feature_names)}")
print("Features:", feature_names[:10], "...")

## Application du pipeline — Régression (CLV)

In [ ]:
from src.preprocessing import build_preprocessor_regression, load_and_split

X_tr_r, X_te_r, y_tr_r, y_te_r, prep_r = preprocess_data(
    '../data/raw/customer_churn_business_dataset.csv', task='regression', save=True
)
print(f"CLV (total_revenue) — mean: {y_tr_r.mean():.1f} | std: {y_tr_r.std():.1f}")
print(f"X_train régression shape: {X_tr_r.shape}")

## Vérification absence de data leakage

In [ ]:
# Les stats du scaler doivent être celles du train uniquement
scaler = preprocessor.named_transformers_['num'].named_steps['scaler']
print("Moyenne du scaler (train):", scaler.mean_[:5].round(3))
print("\nMoyenne X_train_proc (doit être ~0):", X_train_proc[:, :5].mean(axis=0).round(3))
print("Std X_train_proc (doit être ~1):", X_train_proc[:, :5].std(axis=0).round(3))
print("\nMoyenne X_test_proc (peut différer):", X_test_proc[:, :5].mean(axis=0).round(3))